<a href="https://colab.research.google.com/github/hoangchaulanbao/HocmayNC/blob/main/btcn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Khai báo Siêu tham số (Hyper-parameters):
Học viên phải liệt kê rõ ràng và giải thích lý do lựa chọn các siêu tham số: Learning Rate, Batch Size, Epochs, Optimizer (SGD, Adam, Adamax...), và các hàm Loss tương ứng.
# Trực quan hóa quá trình huấn luyện:
Bắt buộc vẽ biểu đồ Accuracy Curve và Loss Curve (Train vs. Validation) qua từng Epoch để chứng minh mô hình hội tụ và phân tích hiện tượng Overfitting/Underfitting.
# Đánh giá mô hình Phân loại:
Đối với các bài toán phân loại (áp dụng cho MLP hoặc CNN), học viên bắt buộc phải thực hiện các yêu cầu sau:
-  Lập và giải thích Ma trận nhầm lẫn (Confusion Matrix) để thể hiện tương quan giữa kết quả thực tế và dự đoán của mô hình.
- Tính toán và diễn giải ý nghĩa thực tế của các chỉ số: Precision, Recall, và F1-Score.
- Phân tích sự đánh đổi (trade-off) giữa Precision và Recall. Bắt buộc minh họa bằng đường cong PR (Precision-
Recall Curve) và giải thích ảnh hưởng của việc thay đổi ngưỡng (threshold).
- Biện luận ngữ cảnh: Biện luận về việc lựa chọn và ưu tiên chỉ số nào (Precision hay Recall) tùy thuộc vào ngữ
cảnh cụ thể của bài toán.

# A. Mô hình Hồi quy (Regression Model)
-  Thuật toán: Triển khai Linear Regression sử dụng Gradient Descent.
- Đánh giá: Sử dụng các chỉ số MAE, MSE, RMSE và $R^2$ Score. Giải thích ý nghĩa của $R^2$ âm (nếu có) so với baseline.

# Ứng dụng :
- Bài toán: Dự đoán giá xe (price)

# PIPELINE HỆ THỐNG:
- BƯỚC 1 Đọc dữ liệu
- BƯỚC 2 EDA + Data Cleaning
- BƯỚC 3 Feature Engineering
- BƯỚC 4 Train/Test Split
- BƯỚC 5 Chuẩn hóa dữ liệu
- BƯỚC 6 Xây dựng mô hình Gradient Descent
- BƯỚC 7 Huấn luyện (Training)
- BƯỚC 8 Đánh giá (MAE, MSE, RMSE, R2)
- BƯỚC 9 Visualization (Loss Curve)
- BƯỚC 10 Phân tích kết quả (Overfitting / Underfitting)

import thư viện

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# ── Matplotlib dark theme ───────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor":  "#0a0e1a",
    "axes.facecolor":    "#111827",
    "axes.edgecolor":    "#1e3a5f",
    "axes.labelcolor":   "#94a3b8",
    "xtick.color":       "#64748b",
    "ytick.color":       "#64748b",
    "text.color":        "#e2e8f0",
    "grid.color":        "#1e293b",
    "grid.linewidth":    0.6,
    "font.family":       "DejaVu Sans",
    "axes.titlesize":    14,
    "axes.labelsize":    12,
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

PALETTE   = ["#00d4ff","#00f5c8","#ff6b9d","#ffb347","#c084fc","#34d399","#f87171"]
BLUE_CYAN = ["#00d4ff","#00f5c8"]
print("✅ Libraries imported.")


✅ Libraries imported.


# BƯỚC 1 : ĐỌC DỮ LIỆU (LOAD DATA)

In [3]:
url = "https://raw.githubusercontent.com/hoangchaulanbao/HocmayNC/main/car_sales_data.csv"
df = pd.read_csv(url)


rows, cols = df.shape
print("Số dòng:", rows)
print("Số cột:", cols)
# xuất 5 dòng đầu
df.head()

Số dòng: 50000
Số cột: 7


,Manufacturer,Model,Engine size,Fuel type,Year of manufacture,Mileage,Price
0,Ford,Fiesta,1.0,Petrol,2002,127300,3074
1,Porsche,718 Cayman,4.0,Petrol,2016,57850,49704
2,Ford,Mondeo,1.6,Diesel,2014,39190,24072
3,Toyota,RAV4,1.8,Hybrid,1988,210814,1705
4,VW,Polo,1.0,Petrol,2006,127869,4101


# BƯỚC 2: KHÁM PHÁ DỮ LIỆU (EDA)

# Kiểm tra Thông tin & loại dữ liệu (data_type)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Manufacturer         50000 non-null  object 
 1   Model                50000 non-null  object 
 2   Engine size          50000 non-null  float64
 3   Fuel type            50000 non-null  object 
 4   Year of manufacture  50000 non-null  int64  
 5   Mileage              50000 non-null  int64  
 6   Price                50000 non-null  int64  
dtypes: float64(1), int64(3), object(3)
memory usage: 2.7+ MB


# Kiểm tra kiểu dữ liệu của từng cột

In [7]:
df.dtypes

,0
Manufacturer,object
Model,object
Engine size,float64
Fuel type,object
Year of manufacture,int64
Mileage,int64
Price,int64


# Kiểm tra số lượng giá trị của mỗi cột

In [8]:
df.nunique()

,0
Manufacturer,5
Model,15
Engine size,14
Fuel type,3
Year of manufacture,39
Mileage,44971
Price,25045


#Kiểm tra  Thống kê của tập dữ liệu

In [5]:
df.describe()

,Engine size,Year of manufacture,Mileage,Price
count,50000.000000,50000.000000,50000.000000,50000.000000
mean,1.773058,2004.209440,112497.320700,13828.903160
std,0.734108,9.645965,71632.515602,16416.681336
min,1.000000,1984.000000,630.000000,76.000000
25%,1.400000,1996.000000,54352.250000,3060.750000
50%,1.600000,2004.000000,100987.500000,7971.500000
75%,2.000000,2012.000000,158601.000000,19026.500000
max,5.000000,2022.000000,453537.000000,168081.000000


# Kiểm tra dữ liệu thiếu (missing)


In [11]:
null_counts = df.isnull().sum()
print(null_counts)
print(f"\n số lượng giá trị thiếu: {null_counts.sum()}")

Manufacturer           0
Model                  0
Engine size            0
Fuel type              0
Year of manufacture    0
Mileage                0
Price                  0
dtype: int64

 số lượng giá trị thiếu: 0


# Kiểu tra dữ liệu trùng (duplicate)

In [13]:
print(f"Số dòng bị trùng: {df.duplicated().sum()}")
df.drop_duplicates(inplace=True)
print(f"Sau khi xòa dòng trùng   → Số dòng còn lại: {df.shape[0]}")

Số dòng bị trùng: 0
Sau khi xòa dòng trùng   → Số dòng còn lại: 49988
